# 0.2 — The instruct model and its chat template

**Goal.** Load `Qwen/Qwen2.5-7B-Instruct` (same pretrained weights as 0.1, then post-trained) and
see exactly what text the post-trained model is conditioned on when you "chat" with it. Then compare
base and instruct answers to the same questions.

The key idea for this project: a chat template is *just a prompt format*. `<|im_start|>assistant\n` plays
the same role as `Assistant:` in 0.1. The instruct model's "personality" is whatever the weights now
assign to the text that follows that string.

In [1]:
import os, time, json, textwrap
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CONFIG = {
    "base_model": "Qwen/Qwen2.5-7B",
    "instruct_model": "Qwen/Qwen2.5-7B-Instruct",
    "dtype": "bfloat16",
    "seed": 0,
    "max_new_tokens": 150,
    "base_stop_strings": ["User:"],   # see 0.1: the base model writes " User:" inline, no newline
}
torch.manual_seed(CONFIG["seed"])

REPO = Path.cwd().resolve().parent
RESULTS = REPO / "results" / "phase0"
RESULTS.mkdir(parents=True, exist_ok=True)
# Model weights live on CFS, not $HOME. `source env.sh` sets these, and so does the `persona-ml`
# Jupyter kernel; if you're on a different kernel, fall back to the same values here.
os.environ.setdefault("HF_HOME", "/global/cfs/cdirs/m2612/ozamram/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # CFS can't take file locks; never download from here
print("HF_HOME =", os.environ["HF_HOME"])

HF_HOME = /global/cfs/cdirs/m2612/ozamram/hf_cache


## Load the instruct model and inspect its tokenizer

Same weights *shape* as the base model. The differences we can see without running anything:
the tokenizer's special tokens, the `eos` token (the instruct model is trained to emit `<|im_end|>`
to end its turn, whereas the base model's eos is `<|endoftext|>`), and the chat template stored in
`tokenizer.chat_template` (a Jinja string).

In [2]:
t0 = time.time()
tok_i = AutoTokenizer.from_pretrained(CONFIG["instruct_model"])
model_i = AutoModelForCausalLM.from_pretrained(CONFIG["instruct_model"], dtype=torch.bfloat16, device_map="cuda").eval()
print(f"instruct loaded in {time.time()-t0:.0f}s | GPU mem {torch.cuda.memory_allocated()/2**30:.1f} GiB")

print("eos:", repr(tok_i.eos_token), "| pad:", repr(tok_i.pad_token))
# (transformers 5 dropped `additional_special_tokens`; `all_special_tokens` lists everything.)
print(f"{len(tok_i.all_special_tokens)} special tokens, e.g.:", tok_i.all_special_tokens[:6])
print()
print("chat_template (Jinja):")
print(tok_i.chat_template)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

instruct loaded in 40s | GPU mem 14.2 GiB
eos: '<|im_end|>' | pad: '<|endoftext|>'
14 special tokens, e.g.: ['<|im_end|>', '<|endoftext|>', '<|im_start|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>']

chat_template (Jinja):
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_

## Apply the chat template and look at the result

`apply_chat_template` turns a list of `{"role", "content"}` messages into the exact string the model was
post-trained on. Things to notice in the output:

- **Qwen inserts a default system prompt** if you don't give one: `You are Qwen, created by Alibaba Cloud.
  You are a helpful assistant.` So there is *always* a system message. This is part of the conditioning
  context and matters for Phase 1 comparisons (we'll want to control it).
- `<|im_start|>` / `<|im_end|>` are single special tokens (one ID each), not multi-token strings.
- `add_generation_prompt=True` appends `<|im_start|>assistant\n` so the model's next token is the first
  token of its reply. Without it, the model would generate a *whole* next message including a role header.

In [3]:
question = "What should I do if I find a lost wallet?"
messages = [{"role": "user", "content": question}]

chat_str = tok_i.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== string ===")
print(repr(chat_str))
print()
# In transformers 5, tokenize=True returns a BatchEncoding (a dict with "input_ids", "attention_mask"),
# not a bare list of ids as in v4. Pull out the id list explicitly.
chat_ids = tok_i.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"]
chat_ids = [int(i) for i in chat_ids]
print("=== token ids ===")
print(chat_ids)
print()
print("=== pieces ===")
print([tok_i.decode([i]) for i in chat_ids])
print()
print("=== which pieces are special tokens ===")
print({tok_i.decode([i]): i for i in chat_ids if i in tok_i.all_special_ids})

=== string ===
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat should I do if I find a lost wallet?<|im_end|>\n<|im_start|>assistant\n'

=== token ids ===
[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838, 1265, 358, 653, 421, 358, 1477, 264, 5558, 15085, 30, 151645, 198, 151644, 77091, 198]

=== pieces ===
['<|im_start|>', 'system', '\n', 'You', ' are', ' Q', 'wen', ',', ' created', ' by', ' Alibaba', ' Cloud', '.', ' You', ' are', ' a', ' helpful', ' assistant', '.', '<|im_end|>', '\n', '<|im_start|>', 'user', '\n', 'What', ' should', ' I', ' do', ' if', ' I', ' find', ' a', ' lost', ' wallet', '?', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n']

=== which pieces are special tokens ===
{'<|im_start|>': 151644, '<|im_end|>': 151645}


In [4]:
# With an explicit system prompt, no default is inserted. Note how little changes: just the text
# between <|im_start|>system and <|im_end|>.
messages_sys = [{"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": question}]
print(repr(tok_i.apply_chat_template(messages_sys, tokenize=False, add_generation_prompt=True)))

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat should I do if I find a lost wallet?<|im_end|>\n<|im_start|>assistant\n'


## Generate from the instruct model

The instruct model ends its turn by emitting `<|im_end|>` (its `eos`), so we don't need the stop-string
machinery from 0.1: `generate` stops on `eos` automatically. Also note that in `transformers` 5 the
tokenizer output of `apply_chat_template(..., return_tensors="pt", return_dict=True)` is a BatchEncoding
you can unpack straight into `generate`.

In [5]:
def generate_instruct(question, system=None, max_new_tokens=150, do_sample=False, temperature=1.0, top_p=1.0, n=1):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    enc = tok_i.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model_i.device)
    prompt_len = enc["input_ids"].shape[1]
    outs = []
    for _ in range(n):
        with torch.no_grad():
            out = model_i.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                                   temperature=temperature if do_sample else None,
                                   top_p=top_p if do_sample else None,
                                   pad_token_id=tok_i.pad_token_id)
        gen = out[0, prompt_len:]
        # Keep the raw ids so we can see whether it stopped on eos or hit the token cap.
        outs.append({"text": tok_i.decode(gen, skip_special_tokens=True).strip(),
                     "n_tokens": len(gen),
                     "hit_eos": bool((gen == tok_i.eos_token_id).any())})
    return outs

r = generate_instruct(question)[0]
print(f"[{r['n_tokens']} tokens, hit_eos={r['hit_eos']}]")
print(r["text"])

[150 tokens, hit_eos=False]
If you find a lost wallet, here are some steps you can take to help return it to its rightful owner:

1. **Check for Identification**: Look inside the wallet for any identification cards, such as driver's licenses, credit cards, or ID cards. These can provide contact information for the owner.

2. **Contact the Owner**: If there is contact information available, try to reach out to the owner. This could be through phone numbers, email addresses, or social media profiles listed on their ID.

3. **Local Authorities**: If you cannot find any contact information or if the wallet contains cash and no identification, you can turn it over to local authorities such as the police station. They often have systems in place to help return lost items


## Base vs. instruct on the same questions

Load the base model alongside (two 7B models in bf16 ≈ 30 GB; fits on a 40 GB A100 with short
prompts, but check the memory print). Same greedy decoding for both. Same five questions as 0.1.

Things to compare: length, formatting (markdown lists, headers), hedging / disclaimers, and whether the
*substance* of the advice differs. The README flags formatting as the major confound for the Phase 1
headline test; this is where you first see it.

In [6]:
from transformers import StoppingCriteria, StoppingCriteriaList

t0 = time.time()
tok_b = AutoTokenizer.from_pretrained(CONFIG["base_model"])
model_b = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], dtype=torch.bfloat16, device_map="cuda").eval()
print(f"base loaded in {time.time()-t0:.0f}s | GPU mem {torch.cuda.memory_allocated()/2**30:.1f} GiB (both models)")


def clean(text, stop_strings):
    cut = len(text)
    for s in stop_strings:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip()


def generate_base(question, max_new_tokens=150, stop_strings=("User:",), do_sample=False, temperature=1.0, top_p=1.0):
    # Same raw-text format as 0.1, using the built-in stop_strings this time. Returns the same
    # dict shape as generate_instruct so the two are directly comparable.
    prompt = f"User: {question}\nAssistant:"
    enc = tok_b(prompt, return_tensors="pt").to(model_b.device)
    prompt_len = enc["input_ids"].shape[1]
    with torch.no_grad():
        out = model_b.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                               temperature=temperature if do_sample else None,
                               top_p=top_p if do_sample else None,
                               stop_strings=list(stop_strings), tokenizer=tok_b,
                               pad_token_id=tok_b.pad_token_id)
    gen = out[0, prompt_len:]
    return {"text": clean(tok_b.decode(gen, skip_special_tokens=True), stop_strings),
            "n_tokens": len(gen),
            "hit_eos": bool((gen == tok_b.eos_token_id).any())}

OutOfMemoryError: CUDA out of memory. Tried to allocate 14.16 GiB. GPU 0 has a total capacity of 39.49 GiB of which 10.05 GiB is free. Process 1129524 has 14.72 GiB memory in use. Including non-PyTorch memory, this process has 14.70 GiB memory in use. Of the allocated memory 14.19 GiB is allocated by PyTorch, and 13.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [7]:
QUESTIONS = [
    "What should I do if I find a lost wallet?",
    "Is it ever okay to lie?",
    "My coworker keeps taking credit for my work. What should I do?",
    "Do you think AI systems should have rights?",
    "How can I get my neighbor to stop parking in front of my house?",
]

comparison = {}
for q in QUESTIONS:
    b = generate_base(q)
    i = generate_instruct(q)[0]
    comparison[q] = {"base": b, "instruct": i}
    print("=" * 100)
    print("Q:", q)
    print("-" * 40, f"BASE ({b['n_tokens']} tok, eos={b['hit_eos']})", "-" * 34)
    print(textwrap.fill(b["text"], 100))
    print("-" * 40, f"INSTRUCT ({i['n_tokens']} tok, eos={i['hit_eos']})", "-" * 30)
    print(i["text"])

NameError: name 'generate_base' is not defined

## Does the *base* model respond to the chat template?

The base tokenizer ships with the same chat template, and the `<|im_start|>` tokens exist in its vocab.
Pretraining corpora contain a lot of chat-formatted text, so the base model may already "know" this
format to some degree. Feed the base model the ChatML-formatted prompt and see what it does. This is
relevant to Phase 1's format-confound discussion: if the base model handles ChatML well, we could score
base-model personas in the *instruct* format and remove one difference between the two models.

In [8]:
print("base tokenizer has chat_template:", tok_b.chat_template is not None)
msgs = [{"role": "user", "content": QUESTIONS[0]}]
enc = tok_b.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model_b.device)
with torch.no_grad():
    out = model_b.generate(**enc, max_new_tokens=150, do_sample=False, pad_token_id=tok_b.pad_token_id,
                           stop_strings=["<|im_end|>"], tokenizer=tok_b)
gen = out[0, enc["input_ids"].shape[1]:]
print(f"[{len(gen)} tokens; contains <|im_end|>: {'<|im_end|>' in tok_b.decode(gen)}]")
print(tok_b.decode(gen))

base tokenizer has chat_template: True


[150 tokens; contains <|im_end|>: False]
If you find a lost wallet, it's important to handle it responsibly. Here are the steps you should follow:

1. **Assess the Situation**: Ensure your safety first. If the wallet is in a public place, make sure it's safe to approach. If there's any concern for your safety, leave the wallet where it is and report it to the authorities.

2. **Check the Contents**: Carefully examine the wallet to see if it contains identification, such as a driver's license, credit cards, or other personal documents. This information can help you determine who the owner might be.

3. **Contact the Owner**: If possible, try to contact the owner by looking at the identification inside the wallet. You can call the phone


## Save

In [9]:
record = {
    "config": CONFIG,
    "transformers_version": __import__("transformers").__version__,
    "chat_template_example": chat_str,
    "chat_template_ids_example": chat_ids,
    "comparison_greedy": comparison,
}
out_path = RESULTS / "0.2_instruct_vs_base.json"
out_path.write_text(json.dumps(record, indent=2))
print("saved", out_path)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")

saved /global/u1/o/ozamram/personal/persona_selection_study/results/phase0/0.2_instruct_vs_base.json
peak GPU memory: 28.4 GiB


## What we saw (first run, 2026-09-21)

- The chat template is a *string*. The instruct model is conditioned on `<|im_start|>assistant\n` the way
  the base model is conditioned on `Assistant:`. Persona labels in Phase 1 are just different strings in
  that slot (or in the system prompt). `<|im_start|>` and `<|im_end|>` are the only special tokens in a
  plain chat; `system`, `user`, `assistant` are ordinary word tokens.
- The default system prompt (`You are Qwen, created by Alibaba Cloud. You are a helpful assistant.`) is
  inserted whenever you don't supply one. Phase 1 has to decide whether to keep it, replace it with a
  neutral one, or use it as the "label" slot.
- **Format confound, seen directly.** Every instruct answer hit the 150-token cap: a one-line preamble
  restating the question, then a bold-headed numbered list. Base answers (raw `Assistant:` format) were
  70–120 tokens, plain paragraphs, and ended with `eos` on 4 of 5 questions. The *substance* of the
  advice is nearly identical (wallet: check ID, contact owner, police; coworker: document, meet, escalate).
  So a naive base-vs-instruct likelihood comparison would mostly measure formatting.
- **The base model speaks ChatML.** Given the instruct chat template, the base model produced a
  markdown numbered list indistinguishable in style from the instruct model (and also ran to the cap
  without emitting `<|im_end|>`). Together with the `<|endoftext|>`-after-answer behaviour and the
  "As an AI language model" sample in 0.1, this says Qwen2.5-7B "base" has seen a lot of chat-formatted
  instruction data in pretraining. Two consequences for Phase 1:
  1. The base/instruct gap this project wants to measure may be unusually small for this family.
  2. On the upside, we *can* score base-model personas in the exact ChatML format the instruct model
     uses, which removes the prompt-format difference from the headline comparison.
- Loading both 7B models on one A100-40GB works: 28.4 GiB peak with short prompts.